In [0]:
#What a Window Function Does
#======================================================================================
#A normal aggregation (Day 6) collapses many rows into one summary row per group. A window function instead keeps every original row, but adds a calculation that looks across a group of related rows — like “what rank is this employee's salary within their department” or “what was the previous order for this customer.” Nothing is collapsed; every row survives, just enriched with group-aware context

from pyspark.sql.functions import col


data = [(1, 101, "2025-01-10", 500),
        (2, 101, "2025-02-15", 700),
        (3, 101, "2025-03-01", 700),   # tie with order 2 on amount, on purpose
        (4, 102, "2025-01-20", 300),
        (5, 102, "2025-01-20", 300)]   # exact duplicate of order 4, on purpose
columns = ["order_id", "customer_id", "order_date", "amount"]

orders = spark.createDataFrame(data, columns)
orders = orders.withColumn("order_date", col("order_date").cast("date"))
orders.display()


In [0]:
# Defining first window spec
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, rank, dense_rank, col

window_spec = Window.partitionBy("customer_id").orderBy(col("order_date").desc())

#Add row_number() and look at the result
ranked = orders.withColumn("row_num", row_number().over(window_spec))
ranked.display()


In [0]:
# Get each customer's latest order only
latest_orders = ranked.filter(col("row_num") == 1)
latest_orders.display()

In [0]:

#A Very Common DQ Use: Finding the Latest Record per Group

# Compare all three ranking functions
ranked_all = orders.withColumn("row_num", row_number().over(window_spec)) \
                    .withColumn("rank", rank().over(window_spec)) \
                    .withColumn("dense_rank", dense_rank().over(window_spec))

ranked_all.display()

# Key differences:
# row_number(): Always sequential (1,2,3...) even for ties - arbitrary order for duplicates
# rank(): Ties get same rank, then SKIPS (1,2,2,4,5...) - leaves gaps after ties
# dense_rank(): Ties get same rank, NO SKIPS (1,2,2,3,4...) - consecutive ranks

In [0]:
# Detecting the duplicates more precisely than Day 6
# Day 6's duplicate pattern groupBy(key).count().filter(count > 1) that tells you duplicate exists. A window function lets you see and act on which specific rows are the duplicates
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, rank, dense_rank, col, lag, lead
window_spec = Window.partitionBy("customer_id", "order_date").orderBy("order_id")

with_row_num = orders.withColumn("row_num", row_number().over(window_spec))
duplicates_only = with_row_num.filter(col("row_num") > 1)
duplicates_only.display()


In [0]:
#lag() and lead() — Comparing to the Previous or Next Row

window_spec = Window.partitionBy("customer_id").orderBy("order_date")

orders.withColumn("previous_order_amount", lag("amount", 1).over(window_spec)).display()


In [0]:
# lead() looks FORWARD to the next row (opposite of lag)
window_spec = Window.partitionBy("customer_id").orderBy("order_date")

orders.withColumn("next_order_amount", lead("amount", 1).over(window_spec)).display()

# Key difference:
# lag("amount", 1) = previous order's amount (look backward)
# lead("amount", 1) = next order's amount (look forward)
# Both return null at the boundaries (first row for lag, last row for lead)

In [0]:
#Putting It All Together — A Realistic Example

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window_spec = Window.partitionBy("customer_id").orderBy(col("order_date").desc())

ranked = orders.withColumn("row_num", row_number().over(window_spec))

latest_order_per_customer = ranked.filter(col("row_num") == 1)
duplicate_orders = ranked.filter(col("row_num") > 1)

latest_order_per_customer.display()
duplicate_orders.display()
